# Capstone Two – Step 4: Preprocessing and Training Data Development  
**Dataset:** `world_top_restaurants_dataset_cleaned.csv`


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Load cleaned dataset
df = pd.read_csv('world_top_restaurants_dataset_cleaned.csv')

df.head()


,restaurant_name,city,country,cuisine_type,michelin_stars,average_price_usd,chef,established_year,seating_capacity,rating,signature_dish,years_operating,price_category,star_category,continent
0,Arpège,Paris,France,French Vegetarian,3,425,Alain Passard,1986,40,4.9,Heirloom Tomatoes,39,Ultra-Luxury,Three Stars,Europe
1,Odette,Singapore,Singapore,French Contemporary,3,320,Julien Royer,2008,32,4.9,Hokkaido Sea Urchin,17,Luxury,Three Stars,Asia
2,Osteria Francescana,Modena,Italy,Italian Contemporary,3,320,Massimo Bottura,1995,28,4.9,Five Ages of Parmigiano,30,Luxury,Three Stars,Europe
3,The French Laundry,Yountville,USA,French,3,390,Thomas Keller,1994,60,4.9,Oysters and Pearls,31,Luxury,Three Stars,North America
4,Alléno Paris,Paris,France,French Haute Cuisine,3,385,Yannick Alléno,2003,55,4.8,Cabbage & Caviar,22,Luxury,Three Stars,Europe


In [2]:

df.shape, df.dtypes


((50, 15),
 restaurant_name       object
 city                  object
 country               object
 cuisine_type          object
 michelin_stars         int64
 average_price_usd      int64
 chef                  object
 established_year       int64
 seating_capacity       int64
 rating               float64
 signature_dish        object
 years_operating        int64
 price_category        object
 star_category         object
 continent             object
 dtype: object)

In [3]:

target = 'rating'

# Columns to exclude from features (IDs / free text fields)
exclude_cols = [target, 'restaurant_name', 'chef', 'signature_dish']

X = df.drop(columns=[c for c in exclude_cols if c in df.columns])
y = df[target].copy()

X.head(), y.head()


(         city    country          cuisine_type  michelin_stars  \
 0       Paris     France     French Vegetarian               3   
 1   Singapore  Singapore   French Contemporary               3   
 2      Modena      Italy  Italian Contemporary               3   
 3  Yountville        USA                French               3   
 4       Paris     France  French Haute Cuisine               3   
 
    average_price_usd  established_year  seating_capacity  years_operating  \
 0                425              1986                40               39   
 1                320              2008                32               17   
 2                320              1995                28               30   
 3                390              1994                60               31   
 4                385              2003                55               22   
 
   price_category star_category      continent  
 0   Ultra-Luxury   Three Stars         Europe  
 1         Luxury   Three St

In [4]:

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_features, categorical_features


(['michelin_stars',
  'average_price_usd',
  'established_year',
  'seating_capacity',
  'years_operating'],
 ['city',
  'country',
  'cuisine_type',
  'price_category',
  'star_category',
  'continent'])

In [5]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['michelin_stars', 'average_price_usd',
                                  'established_year', 'seating_capacity',
                                  'years_operating']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['city', 'country', 'cuisine_type',
                                  'price_category', 'star_category',
                                  'continent'])])

In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((40, 11), (10, 11), (40,), (10,))

In [7]:

# Fit on train
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape


((40, 96), (10, 96))

In [8]:

# Numeric feature names stay the same
num_feature_names = numeric_features

# OneHotEncoder feature names
ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()

feature_names = num_feature_names + cat_feature_names

len(feature_names), feature_names[:15]


(96,
 ['michelin_stars',
  'average_price_usd',
  'established_year',
  'seating_capacity',
  'years_operating',
  'city_Alba',
  'city_Bangkok',
  'city_Barcelona',
  'city_Bogotá',
  'city_Copenhagen',
  'city_Dénia',
  'city_Errenteria',
  'city_Girona',
  'city_Healdsburg',
  'city_Hong Kong'])

In [9]:

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

X_train_df.head()


,michelin_stars,average_price_usd,established_year,seating_capacity,years_operating,city_Alba,city_Bangkok,city_Barcelona,city_Bogotá,city_Copenhagen,...,price_category_Premium,price_category_Ultra-Luxury,star_category_No Stars,star_category_One Star,star_category_Three Stars,star_category_Two Stars,continent_Asia,continent_Europe,continent_North America,continent_South America
12,0.888205,0.170614,-1.550871,1.358339,1.550871,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.888205,1.127331,0.076100,0.881131,-0.076100,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
37,-0.803614,-1.211309,0.600930,-1.095874,-0.600930,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
8,-1.649524,-1.051856,0.338515,0.540268,-0.338515,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.888205,1.180482,-0.396246,1.221994,0.396246,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [10]:

train_out_path = "restaurants_train_preprocessed.csv"
test_out_path = "restaurants_test_preprocessed.csv"

# Save X + y together (helpful for modeling workflows)
train_dev = X_train_df.copy()
train_dev[target] = y_train

test_dev = X_test_df.copy()
test_dev[target] = y_test

train_dev.to_csv(train_out_path, index=False)
test_dev.to_csv(test_out_path, index=False)

train_out_path, test_out_path


('restaurants_train_preprocessed.csv', 'restaurants_test_preprocessed.csv')